In [1]:
import os
import json
import hashlib
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan
from sklearn.cluster import AgglomerativeClustering
from anytree import Node, RenderTree

warnings.filterwarnings("ignore")

d:\git\Taxonomy_Buidling_Textual_Corpora\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_json("D:\git\Taxonomy_Buidling_Textual_Corpora\data\icecat_data_test.json")
df.head(5)

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,ProductSeries.Language,ProductSeries.Value,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names
500081,Fujitsu,,https://images.icecat.biz/img/brand/thumb/15_b...,Fujitsu,https://images.icecat.biz/img/brand/thumb/15_b...,FSP:G-SW3Z560PRE0S,[],788,EN,Warranty & Support Extensions,...,None,None,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...
741063,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,5KM83PA,None,151,EN,Notebooks,...,EN,13,1268560.0,EN,2018-10-21 21:55:51,"[Windows 10 Home 64-bit, Intel® Core™ i7-8565U...","[{'VirtualCategoryID': 329, 'UNCATID': '432115...",NaN,2833>150>151,Computers & Electronics>Computers>Notebooks
1091454,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,4NG33EA,[],153,EN,PCs/Workstations,...,EN,880,NaN,None,None,None,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...
522928,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,83061,[],883,EN,Networking Cables,...,None,None,NaN,None,None,None,None,NaN,2833>830>883,Computers & Electronics>Computer Cables>Networ...
479478,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,5PS0A14091,[],788,EN,Warranty & Support Extensions,...,None,None,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...


In [3]:
df_raw = pd.read_json("D:/git/Taxonomy_Buidling_Textual_Corpora/data/icecat_data_test.json")
list(df_raw.columns)


['Brand',
 'BrandInfo.BrandLocalName',
 'BrandInfo.BrandLogo',
 'BrandInfo.BrandName',
 'BrandLogo',
 'BrandPartCode',
 'BulletPoints',
 'Category.CategoryID',
 'Category.Name.Language',
 'Category.Name.Value',
 'Description.Disclaimer',
 'Description.ID',
 'Description.LeafletPDFURL',
 'Description.LongDesc',
 'Description.LongProductName',
 'Description.ManualPDFSize',
 'Description.ManualPDFURL',
 'Description.MiddleDesc',
 'Description.PDFSize',
 'Description.URL',
 'Description.Updated',
 'EndOfLifeDate',
 'GTIN',
 'IcecatId',
 'ProductFamily.ProductFamilyID',
 'ProductName',
 'ProductSeries.SeriesID',
 'ReleaseDate',
 'SummaryDescription.LongSummaryDescription',
 'SummaryDescription.ShortSummaryDescription',
 'Title',
 'Description.WarrantyInfo',
 'Description',
 'ProductFamily.Language',
 'ProductFamily.Value',
 'ProductSeries.Language',
 'ProductSeries.Value',
 'BulletPoints.BulletPointsId',
 'BulletPoints.Language',
 'BulletPoints.Updated',
 'BulletPoints.Values',
 'VirtualCat

In [4]:
i = 0   # change this to inspect other rows
for col in df_raw.columns:
    print(f"\n=== {col} ===")
    print(df_raw.iloc[i][col])



=== Brand ===
Fujitsu

=== BrandInfo.BrandLocalName ===


=== BrandInfo.BrandLogo ===
https://images.icecat.biz/img/brand/thumb/15_bcd6550338dd47a4bb4df7d2888c5fd0.jpg

=== BrandInfo.BrandName ===
Fujitsu

=== BrandLogo ===
https://images.icecat.biz/img/brand/thumb/15_bcd6550338dd47a4bb4df7d2888c5fd0.jpg

=== BrandPartCode ===
FSP:G-SW3Z560PRE0S

=== BulletPoints ===
[]

=== Category.CategoryID ===
788

=== Category.Name.Language ===
EN

=== Category.Name.Value ===
Warranty & Support Extensions

=== Description.Disclaimer ===


=== Description.ID ===
56731228.0

=== Description.LeafletPDFURL ===
https://objects.icecat.biz/objects/mmo_36392790_1494417255_3073_28919.pdf

=== Description.LongDesc ===
In times of growing complexity and decreasing investments, ensuring early IT planning is essential for setting up a reliable infrastructure. Since inefficient processes and long downtimes can result in tremendous losses, companies need a clear offering that fulfills their business and IT nee

In [5]:
# Work on a copy and (for now) limit to 1000 rows
df = df_raw.copy()
df.head()

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,ProductSeries.Language,ProductSeries.Value,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names
500081,Fujitsu,,https://images.icecat.biz/img/brand/thumb/15_b...,Fujitsu,https://images.icecat.biz/img/brand/thumb/15_b...,FSP:G-SW3Z560PRE0S,[],788,EN,Warranty & Support Extensions,...,None,None,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...
741063,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,5KM83PA,None,151,EN,Notebooks,...,EN,13,1268560.0,EN,2018-10-21 21:55:51,"[Windows 10 Home 64-bit, Intel® Core™ i7-8565U...","[{'VirtualCategoryID': 329, 'UNCATID': '432115...",NaN,2833>150>151,Computers & Electronics>Computers>Notebooks
1091454,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,4NG33EA,[],153,EN,PCs/Workstations,...,EN,880,NaN,None,None,None,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...
522928,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,83061,[],883,EN,Networking Cables,...,None,None,NaN,None,None,None,None,NaN,2833>830>883,Computers & Electronics>Computer Cables>Networ...
479478,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,5PS0A14091,[],788,EN,Warranty & Support Extensions,...,None,None,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...


In [6]:
# Use the first 1000 rows to prototype the pipeline
df = df.head(1000).reset_index(drop=True)
print("Using rows:", len(df))

Using rows: 1000


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 45 columns):
 #   Column                                      Non-Null Count  Dtype  
---  ------                                      --------------  -----  
 0   Brand                                       1000 non-null   object 
 1   BrandInfo.BrandLocalName                    1000 non-null   object 
 2   BrandInfo.BrandLogo                         1000 non-null   object 
 3   BrandInfo.BrandName                         1000 non-null   object 
 4   BrandLogo                                   1000 non-null   object 
 5   BrandPartCode                               1000 non-null   object 
 6   BulletPoints                                914 non-null    object 
 7   Category.CategoryID                         1000 non-null   int64  
 8   Category.Name.Language                      1000 non-null   object 
 9   Category.Name.Value                         1000 non-null   object 
 10  Description.D

In [8]:
#  Columns that carry product meaning
TEXT_COLS = [
    "Brand",
    "ProductName",
    "Title",
    "Description.LongProductName",
    "Description.LongDesc",
    "SummaryDescription.LongSummaryDescription",
    "SummaryDescription.ShortSummaryDescription",
    "Category.Name.Value",
    "pathlist_names",
]

In [9]:
def to_text(x):
    """
    Convert a value to a clean string:
    - handle lists
    - avoid 'nan' / 'None' noise
    """
    if isinstance(x, list):
        x = " ".join(map(str, x))
    if x is None:
        return ""
    x = str(x)
    if x.lower() in ["none", "nan"]:
        return ""
    return x

In [10]:
def build_metadata_text(row):
    """
    Merge all meaningful fields into one big 'raw' text per product.
    Skip empty parts.
    """
    parts = [to_text(row.get(col, "")) for col in TEXT_COLS]
    return " ".join(p for p in parts if p)

df["metadata_text"] = df.apply(build_metadata_text, axis=1)

In [11]:
df.head()

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,ProductSeries.Value,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names,metadata_text
0,Fujitsu,,https://images.icecat.biz/img/brand/thumb/15_b...,Fujitsu,https://images.icecat.biz/img/brand/thumb/15_b...,FSP:G-SW3Z560PRE0S,[],788,EN,Warranty & Support Extensions,...,None,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...,Fujitsu FSP:G-SW3Z560PRE0S Fujitsu FSP:G-SW3Z5...
1,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,5KM83PA,None,151,EN,Notebooks,...,13,1268560.0,EN,2018-10-21 21:55:51,"[Windows 10 Home 64-bit, Intel® Core™ i7-8565U...","[{'VirtualCategoryID': 329, 'UNCATID': '432115...",NaN,2833>150>151,Computers & Electronics>Computers>Notebooks,HP 13-ap0023tu HP Spectre x360 13-ap0023tu Blu...
2,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,4NG33EA,[],153,EN,PCs/Workstations,...,880,NaN,None,None,None,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...,HP 880-156nf HP OMEN 880-156nf 8th gen Intel® ...
3,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,83061,[],883,EN,Networking Cables,...,None,NaN,None,None,None,None,NaN,2833>830>883,Computers & Electronics>Computer Cables>Networ...,C2G 1m Cat5e Non-Booted Unshielded (UTP) Netwo...
4,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,5PS0A14091,[],788,EN,Warranty & Support Extensions,...,None,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...,Lenovo 5PS0A14091 Lenovo 5PS0A14091 warranty/s...


In [13]:
# Clean the merged text into 'metadata_text_clean'
import re
def clean_text(s: str) -> str:
    s = str(s)

    # a) lowercase
    s = s.lower()

    # b) remove HTML tags like <br>, <b>, etc.
    s = re.sub(r"<[^>]+>", " ", s)

    # c) keep letters, digits, spaces, and these symbols:
    #    - '-' (rtx-4060, 5km83pa)
    #    - '+' (iphone 14+)
    #    - 'x' (3840x2160)
    #    - '/' (802.11n/ac)
    s = re.sub(r"[^a-z0-9\-+x/ ]+", " ", s)

    # d) collapse multiple spaces
    s = re.sub(r"\s+", " ", s).strip()

    return s

df["metadata_text_clean"] = df["metadata_text"].apply(clean_text)

In [14]:
df.head()

,Brand,BrandInfo.BrandLocalName,BrandInfo.BrandLogo,BrandInfo.BrandName,BrandLogo,BrandPartCode,BulletPoints,Category.CategoryID,Category.Name.Language,Category.Name.Value,...,BulletPoints.BulletPointsId,BulletPoints.Language,BulletPoints.Updated,BulletPoints.Values,VirtualCategory,SummaryDescription,pathlist_ids,pathlist_names,metadata_text,metadata_text_clean
0,Fujitsu,,https://images.icecat.biz/img/brand/thumb/15_b...,Fujitsu,https://images.icecat.biz/img/brand/thumb/15_b...,FSP:G-SW3Z560PRE0S,[],788,EN,Warranty & Support Extensions,...,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...,Fujitsu FSP:G-SW3Z560PRE0S Fujitsu FSP:G-SW3Z5...,fujitsu fsp g-sw3z560pre0s fujitsu fsp g-sw3z5...
1,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,5KM83PA,None,151,EN,Notebooks,...,1268560.0,EN,2018-10-21 21:55:51,"[Windows 10 Home 64-bit, Intel® Core™ i7-8565U...","[{'VirtualCategoryID': 329, 'UNCATID': '432115...",NaN,2833>150>151,Computers & Electronics>Computers>Notebooks,HP 13-ap0023tu HP Spectre x360 13-ap0023tu Blu...,hp 13-ap0023tu hp spectre x360 13-ap0023tu blu...
2,HP,,https://images.icecat.biz/img/brand/thumb/1_91...,HP,https://images.icecat.biz/img/brand/thumb/1_91...,4NG33EA,[],153,EN,PCs/Workstations,...,NaN,None,None,None,"[{'VirtualCategoryID': 195, 'UNCATID': '431718...",NaN,2833>150>153,Computers & Electronics>Computers>PCs/Workstat...,HP 880-156nf HP OMEN 880-156nf 8th gen Intel® ...,hp 880-156nf hp omen 880-156nf 8th gen intel c...
3,C2G,,https://images.icecat.biz/img/brand/thumb/2834...,C2G,https://images.icecat.biz/img/brand/thumb/2834...,83061,[],883,EN,Networking Cables,...,NaN,None,None,None,None,NaN,2833>830>883,Computers & Electronics>Computer Cables>Networ...,C2G 1m Cat5e Non-Booted Unshielded (UTP) Netwo...,c2g 1m cat5e non-booted unshielded utp network...
4,Lenovo,,https://images.icecat.biz/img/brand/thumb/728_...,Lenovo,https://images.icecat.biz/img/brand/thumb/728_...,5PS0A14091,[],788,EN,Warranty & Support Extensions,...,NaN,None,None,None,None,NaN,2833>839>788,Computers & Electronics>Warranty & Support>War...,Lenovo 5PS0A14091 Lenovo 5PS0A14091 warranty/s...,lenovo 5ps0a14091 lenovo 5ps0a14091 warranty/s...


In [15]:
df[['metadata_text','metadata_text_clean']]

,metadata_text,metadata_text_clean
0,Fujitsu FSP:G-SW3Z560PRE0S Fujitsu FSP:G-SW3Z5...,fujitsu fsp g-sw3z560pre0s fujitsu fsp g-sw3z5...
1,HP 13-ap0023tu HP Spectre x360 13-ap0023tu Blu...,hp 13-ap0023tu hp spectre x360 13-ap0023tu blu...
2,HP 880-156nf HP OMEN 880-156nf 8th gen Intel® ...,hp 880-156nf hp omen 880-156nf 8th gen intel c...
3,C2G 1m Cat5e Non-Booted Unshielded (UTP) Netwo...,c2g 1m cat5e non-booted unshielded utp network...
4,Lenovo 5PS0A14091 Lenovo 5PS0A14091 warranty/s...,lenovo 5ps0a14091 lenovo 5ps0a14091 warranty/s...
...,...,...
995,HP HF9Q5E HP HF9Q5E warranty/support extension...,hp hf9q5e hp hf9q5e warranty/support extension...
996,Tripp Lite Duplex Multimode 62.5/125 Fiber Ada...,tripp lite duplex multimode 62 5/125 fiber ada...
997,Philips In-Ear Headphones SHE8100RD/27 Philips...,philips in-ear headphones she8100rd/27 philips...
998,Toshiba SIC1029729LCD0 Toshiba SIC1029729LCD0 ...,toshiba sic1029729lcd0 toshiba sic1029729lcd0 ...


In [16]:
# Quick sanity check
print("\nRAW MERGED TEXT (first 300 chars):\n")
print(df["metadata_text"].iloc[0][:300])

print("\nCLEANED TEXT (first 300 chars):\n")
print(df["metadata_text_clean"].iloc[0][:300])


RAW MERGED TEXT (first 300 chars):

Fujitsu FSP:G-SW3Z560PRE0S Fujitsu FSP:G-SW3Z560PRE0S warranty/support extension SP 3y TS Sub & Upgr, 9x5, 4h Rm Rt f/ CS200c Advanced 4TB, EMEIA In times of growing complexity and decreasing investments, ensuring early IT planning is essential for setting up a reliable infrastructure. Since ineffic

CLEANED TEXT (first 300 chars):

fujitsu fsp g-sw3z560pre0s fujitsu fsp g-sw3z560pre0s warranty/support extension sp 3y ts sub upgr 9x5 4h rm rt f/ cs200c advanced 4tb emeia in times of growing complexity and decreasing investments ensuring early it planning is essential for setting up a reliable infrastructure since inefficient pr


In [19]:
KEEP_COLS = [
    "Brand",
    "ProductName",
    "Title",
    "Description.LongProductName",
    "Description.LongDesc",
    "SummaryDescription.LongSummaryDescription",
    "SummaryDescription.ShortSummaryDescription",
    "metadata_text",
    "metadata_text_clean",
    "pathlist_names",     #  useful for evaluation
    "pathlist_ids",       #  useful for evaluation
]

In [20]:
# keep only columns that actually exist in df
KEEP_COLS = [c for c in KEEP_COLS if c in df.columns]

df_clean = df[KEEP_COLS].copy()

print("Columns kept:", df_clean.columns.tolist())
print("Shape:", df_clean.shape)

df_clean.head(3)

Columns kept: ['Brand', 'ProductName', 'Title', 'Description.LongProductName', 'Description.LongDesc', 'SummaryDescription.LongSummaryDescription', 'SummaryDescription.ShortSummaryDescription', 'metadata_text', 'metadata_text_clean', 'pathlist_names', 'pathlist_ids']
Shape: (1000, 11)


,Brand,ProductName,Title,Description.LongProductName,Description.LongDesc,SummaryDescription.LongSummaryDescription,SummaryDescription.ShortSummaryDescription,metadata_text,metadata_text_clean,pathlist_names,pathlist_ids
0,Fujitsu,FSP:G-SW3Z560PRE0S,Fujitsu FSP:G-SW3Z560PRE0S warranty/support ex...,"SP 3y TS Sub & Upgr, 9x5, 4h Rm Rt f/ CS200c A...",In times of growing complexity and decreasing ...,Fujitsu FSP:G-SW3Z560PRE0S. Number of years: 3...,"Fujitsu FSP:G-SW3Z560PRE0S, 3 year(s), 9x5",Fujitsu FSP:G-SW3Z560PRE0S Fujitsu FSP:G-SW3Z5...,fujitsu fsp g-sw3z560pre0s fujitsu fsp g-sw3z5...,Computers & Electronics>Warranty & Support>War...,2833>839>788
1,HP,13-ap0023tu,"HP Spectre x360 13-ap0023tu Blue,Silver Hybrid...","Intel® Core™ i7-8565U (1.8 GHz), 16GB DDR4-SDR...",<b>Revolutionary battery life on a convertible...,HP Spectre x360 13-ap0023tu. Product type: Hyb...,"HP Spectre x360 13-ap0023tu, 8th gen Intel® Co...",HP 13-ap0023tu HP Spectre x360 13-ap0023tu Blu...,hp 13-ap0023tu hp spectre x360 13-ap0023tu blu...,Computers & Electronics>Computers>Notebooks,2833>150>151
2,HP,880-156nf,HP OMEN 880-156nf 8th gen Intel® Core™ i7 i7-8...,None,None,HP OMEN 880-156nf. Processor frequency: 3.2 GH...,"HP OMEN 880-156nf, 3.2 GHz, 8th gen Intel® Cor...",HP 880-156nf HP OMEN 880-156nf 8th gen Intel® ...,hp 880-156nf hp omen 880-156nf 8th gen intel c...,Computers & Electronics>Computers>PCs/Workstat...,2833>150>153


In [21]:
df_clean["metadata_text_clean"]


0      fujitsu fsp g-sw3z560pre0s fujitsu fsp g-sw3z5...
1      hp 13-ap0023tu hp spectre x360 13-ap0023tu blu...
2      hp 880-156nf hp omen 880-156nf 8th gen intel c...
3      c2g 1m cat5e non-booted unshielded utp network...
4      lenovo 5ps0a14091 lenovo 5ps0a14091 warranty/s...
                             ...                        
995    hp hf9q5e hp hf9q5e warranty/support extension...
996    tripp lite duplex multimode 62 5/125 fiber ada...
997    philips in-ear headphones she8100rd/27 philips...
998    toshiba sic1029729lcd0 toshiba sic1029729lcd0 ...
999    hp 15-cx0120t hp pavilion gaming 15-cx0120t bl...
Name: metadata_text_clean, Length: 1000, dtype: object

### 3. Generating semantic embeddings with Sentence-BERT

In this step, I convert each product description into a dense vector using a pre-trained Sentence-BERT model.

1. **Model choice**  
   I use the `all-MiniLM-L6-v2` model from the `sentence-transformers` library. This model produces **384-dimensional** sentence embeddings and is a good trade-off between speed and quality.

2. **Input text**  
   For each product, I take the cleaned text from the `metadata_text_clean` column. This text is already normalized (lowercased, no HTML, minimal punctuation), which makes it suitable for embedding.

3. **Batch encoding**  
   I call `model.encode()` on the list of cleaned texts with:
   - `batch_size=256` to speed up processing on the GPU/CPU
   - `normalize_embeddings=True` so that each embedding has unit length, which is useful for cosine-based similarity and for downstream methods like UMAP.

4. **Result**  
   The output is a NumPy array of shape `(N, 384)`, where `N` is the number of products (here up to 1000). Each row is a semantic representation of one product description. Products with similar meaning lie close together in this high-dimensional space.


In [22]:
from sentence_transformers import SentenceTransformer
import numpy as np

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
print("Loading SBERT model:", MODEL_NAME)
sbert = SentenceTransformer(MODEL_NAME)

texts = df_clean["metadata_text_clean"].fillna("").tolist()

embeddings = sbert.encode(
    texts,
    batch_size=256,
    normalize_embeddings=True,   # important for cosine similarity
    show_progress_bar=True
)

embeddings = np.asarray(embeddings)
embeddings.shape


Loading SBERT model: sentence-transformers/all-MiniLM-L6-v2


Batches: 100%|██████████| 4/4 [00:31<00:00,  7.94s/it]


(1000, 384)

### 4. Dimensionality reduction with UMAP

The Sentence-BERT embeddings live in a 384-dimensional space, which is relatively high-dimensional for clustering. To make clustering more stable and efficient, I reduce the dimensionality using UMAP.

1. **Why UMAP?**  
   UMAP (Uniform Manifold Approximation and Projection) is a non-linear dimensionality reduction technique that tries to preserve the local neighborhood structure of the data. It is well suited for embedding spaces and often works better than PCA on highly non-linear manifolds.

2. **UMAP configuration**  
   I use the following settings:
   - `n_neighbors=15`: how many neighbors UMAP considers to build the local structure
   - `min_dist=0.0`: allows tight clusters in the low-dimensional space
   - `n_components=25`: the target dimensionality (384 → 25)
   - `metric="cosine"`: distance measure that works well with normalized SBERT embeddings
   - `random_state=42`: fixed seed for reproducibility

3. **Result**  
   UMAP maps each 384-dimensional embedding to a 25-dimensional vector. The resulting array has shape `(N, 25)`. This reduced representation keeps semantic neighborhoods while making clustering faster and more robust.


In [23]:
from umap import UMAP

umap = UMAP(
    n_neighbors=15,
    min_dist=0.0,
    n_components=25,
    metric="cosine",
    random_state=42
)

reduced = umap.fit_transform(embeddings)
reduced.shape


(1000, 25)

In [24]:
import hdbscan

clusterer_C = hdbscan.HDBSCAN(
    min_cluster_size=10,
    min_samples=1,
    metric="euclidean"
)

df_clean["C_id"] = clusterer_C.fit_predict(reduced)

unique_clusters = sorted(df_clean["C_id"].unique())
coverage = (df_clean["C_id"] != -1).mean()

print("Unique C_ids:", unique_clusters[:20], "...")
print("Clusters (excluding noise):", len([c for c in unique_clusters if c != -1]))
print("Coverage:", round(coverage, 3))

df_clean["C_id"].value_counts().head()


Unique C_ids: [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18] ...
Clusters (excluding noise): 38
Coverage: 0.918


C_id
 0     92
-1     82
 25    67
 9     61
 12    47
Name: count, dtype: int64

In [25]:
df_clean["C_id"]


0       3
1      37
2      31
3      23
4       4
       ..
995     7
996    22
997    25
998     5
999    -1
Name: C_id, Length: 1000, dtype: int64

In [26]:
def make_evidence(subdf, max_chars=1000):
    """
    Build an evidence blob for a cluster.
    Uses up to 30 product descriptions (cleaned text).
    """
    texts = subdf["metadata_text_clean"].dropna().tolist()

    if len(texts) == 0:
        return ""

    # sample max 30 texts for the blob
    sample_texts = texts[:30] if len(texts) < 30 else random.sample(texts, 30)

    blob = " ".join(sample_texts)
    return blob[:max_chars]


In [48]:
def run_llm(prompt: str, model: str = "llama3.2") -> str:
    prompt = textwrap.dedent(prompt).strip()
    result = subprocess.run(
        ["ollama", "run", model],
        input=prompt.encode("utf-8"),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )

    # Only print real errors (ignore empty lines)
    stderr_text = result.stderr.decode("utf-8", errors="ignore").strip()
    if stderr_text:
        print("⚠ Ollama stderr:", stderr_text)

    return result.stdout.decode("utf-8", errors="ignore")


In [28]:
import hashlib

label_cache = {}

def cached_llm_label(prompt, max_words=4, model="llama3.2"):
    key = hashlib.sha256(prompt.encode()).hexdigest()

    if key in label_cache:
        return label_cache[key]

    raw = run_llm(prompt, model=model)
    # take only first line, trim to max_words
    label = " ".join(raw.strip().split()[:max_words])

    label_cache[key] = label
    return label


In [29]:
import random

C_name_map = {}

unique_cids = sorted(df_clean["C_id"].unique())

for cid in unique_cids:
    if cid == -1:    # skip noise
        continue
        
    subdf = df_clean[df_clean["C_id"] == cid]
    blob = make_evidence(subdf, max_chars=1000)

    prompt = f"""
    You are an expert in product taxonomy.
    Based on the following product descriptions, assign a concise category name (max 4 words).
    ONLY return the category name.

    {blob}
    """

    label = cached_llm_label(prompt, max_words=4)
    C_name_map[cid] = label

    print(f"C_id {cid} → {label}")


⚠ Ollama stderr: ⠙ ⠙ ⠸ ⠼ ⠼ ⠦ ⠦ ⠇ ⠇ ⠏ ⠙ ⠙ ⠹ ⠼ ⠼ ⠴ ⠧ ⠇ ⠏ ⠋ ⠙ ⠙ ⠹ ⠼ ⠼ ⠦ ⠧ ⠧ ⠇ ⠏ ⠙ ⠙ ⠸ ⠼ ⠼ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠸ ⠴ ⠴ ⠦ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠼ ⠦ ⠧ ⠧ ⠇ ⠋ ⠋ ⠙ ⠹ ⠸ ⠼ ⠦ ⠧ ⠇ ⠏ ⠏ ⠙ ⠙ ⠸ ⠼ ⠼ ⠦ ⠧ ⠧ ⠏ ⠋ ⠙ ⠹ ⠸ ⠸ ⠴ ⠴ ⠦ ⠇ ⠏ ⠋ ⠙ ⠹ 
C_id 0 → Canon QY6-0073 Print Head
⚠ Ollama stderr: ⠙ ⠹ ⠹ 
C_id 1 → Antivirus Security Software
⚠ Ollama stderr: ⠙ ⠹ ⠸ 
C_id 2 → Web Security Firewall
⚠ Ollama stderr: ⠙ ⠹ ⠹ 
C_id 3 → Fujitsu Support Services
⚠ Ollama stderr: ⠙ ⠹ ⠸ 
C_id 4 → Lenovo Support Services
⚠ Ollama stderr: ⠙ ⠙ ⠸ 
C_id 5 → Toshiba LCD Screen Replacement
⚠ Ollama stderr: ⠋ ⠹ ⠹ ⠼ 
C_id 6 → Flip Phone Cases
⚠ Ollama stderr: ⠙ ⠙ ⠸ 
C_id 7 → HPE Foundation Care Services
⚠ Ollama stderr: ⠙ ⠹ ⠹ ⠼ 
C_id 8 → HP Computer Hardware Support
⚠ Ollama stderr: ⠙ ⠙ ⠸ ⠼ 
C_id 9 → Laptop LCD Screen Replacement
⚠ Ollama stderr: ⠋ ⠹ ⠹ ⠼ 
C_id 10 → Laptop Replacement Batteries
⚠ Ollama stderr: ⠙ ⠹ ⠸ ⠼ 
C_id 11 → Acer Notebook Motherboard Spares
⚠ Ollama stderr: ⠙ ⠙ ⠹ ⠸ 
C_id 12 → Seagate NAS Drive
⚠ Ollama stderr: ⠙ ⠙ ⠸ ⠼ 
C_id 13 → E

In [30]:
df_clean["C_name"] = df_clean["C_id"].map(C_name_map)
df_clean.head()


,Brand,ProductName,Title,Description.LongProductName,Description.LongDesc,SummaryDescription.LongSummaryDescription,SummaryDescription.ShortSummaryDescription,metadata_text,metadata_text_clean,pathlist_names,pathlist_ids,C_id,C_name
0,Fujitsu,FSP:G-SW3Z560PRE0S,Fujitsu FSP:G-SW3Z560PRE0S warranty/support ex...,"SP 3y TS Sub & Upgr, 9x5, 4h Rm Rt f/ CS200c A...",In times of growing complexity and decreasing ...,Fujitsu FSP:G-SW3Z560PRE0S. Number of years: 3...,"Fujitsu FSP:G-SW3Z560PRE0S, 3 year(s), 9x5",Fujitsu FSP:G-SW3Z560PRE0S Fujitsu FSP:G-SW3Z5...,fujitsu fsp g-sw3z560pre0s fujitsu fsp g-sw3z5...,Computers & Electronics>Warranty & Support>War...,2833>839>788,3,Fujitsu Support Services
1,HP,13-ap0023tu,"HP Spectre x360 13-ap0023tu Blue,Silver Hybrid...","Intel® Core™ i7-8565U (1.8 GHz), 16GB DDR4-SDR...",<b>Revolutionary battery life on a convertible...,HP Spectre x360 13-ap0023tu. Product type: Hyb...,"HP Spectre x360 13-ap0023tu, 8th gen Intel® Co...",HP 13-ap0023tu HP Spectre x360 13-ap0023tu Blu...,hp 13-ap0023tu hp spectre x360 13-ap0023tu blu...,Computers & Electronics>Computers>Notebooks,2833>150>151,37,HP Spectre X360 Laptop
2,HP,880-156nf,HP OMEN 880-156nf 8th gen Intel® Core™ i7 i7-8...,None,None,HP OMEN 880-156nf. Processor frequency: 3.2 GH...,"HP OMEN 880-156nf, 3.2 GHz, 8th gen Intel® Cor...",HP 880-156nf HP OMEN 880-156nf 8th gen Intel® ...,hp 880-156nf hp omen 880-156nf 8th gen intel c...,Computers & Electronics>Computers>PCs/Workstat...,2833>150>153,31,HP Omen Gaming PC
3,C2G,1m Cat5e Non-Booted Unshielded (UTP) Network P...,C2G 1m Cat5e Non-Booted Unshielded (UTP) Netwo...,Cat5E Assembled UTP Patch Cable Green 1m,Perfect for your home office or a large instal...,C2G 1m Cat5e Non-Booted Unshielded (UTP) Netwo...,C2G 1m Cat5e Non-Booted Unshielded (UTP) Netwo...,C2G 1m Cat5e Non-Booted Unshielded (UTP) Netwo...,c2g 1m cat5e non-booted unshielded utp network...,Computers & Electronics>Computer Cables>Networ...,2833>830>883,23,Network Patch Cables
4,Lenovo,5PS0A14091,Lenovo 5PS0A14091 warranty/support extension,3YR Onsite + Keep Your Drive,Lenovo offers a comprehensive portfolio of val...,"Lenovo 5PS0A14091. Number of years: 3 year(s),...","Lenovo 5PS0A14091, 3 year(s), On-site, 24x7, N...",Lenovo 5PS0A14091 Lenovo 5PS0A14091 warranty/s...,lenovo 5ps0a14091 lenovo 5ps0a14091 warranty/s...,Computers & Electronics>Warranty & Support>War...,2833>839>788,4,Lenovo Support Services


In [ ]:
#confirm of mapping
df_clean[["Brand", "ProductName", "C_id", "C_name"]].head(10)


,Brand,ProductName,C_id,C_name
0,Fujitsu,FSP:G-SW3Z560PRE0S,3,Fujitsu Support Services
1,HP,13-ap0023tu,37,HP Spectre X360 Laptop
2,HP,880-156nf,31,HP Omen Gaming PC
3,C2G,1m Cat5e Non-Booted Unshielded (UTP) Network P...,23,Network Patch Cables
4,Lenovo,5PS0A14091,4,Lenovo Support Services
5,Lenovo,FRU75Y5407,12,Seagate NAS Drive
6,Sony,MDR-E11LP/L,25,Data projector solutions
7,Lexmark,500 Sheet Tray,0,Canon QY6-0073 Print Head
8,Mobiparts,Excellent Wallet Case 2.0 Apple iPhone XS Max ...,6,Flip Phone Cases
9,Acer,E5-573-C14K,27,Acer Aspire E Series


In [37]:
import numpy as np

# get list of valid (non-noise) C_ids
valid_cids = [c for c in df_clean["C_id"].unique() if c != -1]

# compute centroids of each C cluster in UMAP space (`reduced`)
C_centroids = np.vstack([
    reduced[df_clean["C_id"] == c].mean(axis=0)
    for c in valid_cids
])

print("Centroid shape:", C_centroids.shape)


Centroid shape: (38, 25)


In [38]:
from sklearn.cluster import AgglomerativeClustering

agg_B = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=1.0,  # controls how many B clusters are formed
    linkage="ward"
)

B_ids = agg_B.fit_predict(C_centroids)

# Create mapping: C_id → B_id
C_to_B = dict(zip(valid_cids, B_ids))

# Add B_id column to df
df_clean["B_id"] = df_clean["C_id"].map(lambda c: C_to_B.get(c, -1))

df_clean[["C_id", "C_name", "B_id"]].head()


,C_id,C_name,B_id
0,3,Fujitsu Support Services,19
1,37,HP Spectre X360 Laptop,10
2,31,HP Omen Gaming PC,8
3,23,Network Patch Cables,2
4,4,Lenovo Support Services,3


In [39]:
df_clean["B_id"].nunique()

27

In [40]:
df_clean.groupby("B_id")["C_name"].unique()

B_id
-1                                                 [nan]
 0     [Lenovo ThinkPad Keyboard, Ergonomic USB Keybo...
 1       [Fujitsu Lifebook Laptop, Lenovo All-in-One PC]
 2     [Network Patch Cables, Fibre Optic Patch Cable...
 3     [Lenovo Support Services, HP Computer Hardware...
 4     [Here are the assigned, HP Pavilion Keyboard S...
 5                        [HPE Foundation Care Services]
 6     [Acer Aspire E Series, Asus Laptops 6th Gen, T...
 7                              [Digital Compact Camera]
 8       [HP Omen Gaming PC, Toshiba Satellite Notebook]
 9                    [Acer Notebook Motherboard Spares]
 10    [HP Spectre X360 Laptop, HP Pavilion Notebook ...
 11                      [Dell Mission Critical Support]
 12                       [Laptop Replacement Batteries]
 13                                  [Seagate NAS Drive]
 14                           [Data projector solutions]
 15                          [Canon QY6-0073 Print Head]
 16                       

In [41]:
def label_group(df, id_col, name_col, level="B", max_words=4, model="llama3.2"):
    """
    For each group id (e.g. B_id), collect its child names (e.g. C_name)
    and ask the LLM for a shared parent category name.
    """
    name_map = {}
    for gid in sorted(df[id_col].dropna().unique()):
        if gid == -1:
            continue

        child_names = df[df[id_col] == gid][name_col].dropna().unique().tolist()
        if not child_names:
            continue

        # Build a short prompt from the child category names
        examples = ", ".join(child_names[:10])  # limit to 10 for readability

        prompt = f"""
        You are an expert in product taxonomy.

        These are subcategory names at level {level}:
        {examples}

        Suggest a concise parent category name (max {max_words} words)
        that best describes all of them.
        ONLY return the category name.
        """

        label = cached_llm_label(prompt, max_words=max_words, model=model)
        name_map[gid] = label
        print(f"{level}_id {gid} → {label}")

    return name_map


In [42]:
B_name_map = label_group(df_clean, id_col="B_id", name_col="C_name", level="B")
df_clean["B_name"] = df_clean["B_id"].map(B_name_map)

df_clean[["C_id", "C_name", "B_id", "B_name"]].head(15)


⚠ Ollama stderr: ⠙ ⠹ ⠸ ⠼ ⠴ ⠴ ⠦ ⠇ ⠇ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠴ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠏ ⠙ ⠹ ⠹ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠋ ⠙ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠼ ⠴ ⠦ ⠧ ⠇ ⠏ ⠋ ⠙ ⠹ ⠸ ⠸ ⠴ ⠦ ⠧ ⠇ ⠇ ⠏ ⠙ ⠹ ⠹ ⠼ ⠼ ⠦ ⠧ 
B_id 0 → Portable Computer Keyboards
⚠ Ollama stderr: ⠋ ⠙ ⠹ 
B_id 1 → Desktop and Laptops
⚠ Ollama stderr: ⠋ ⠹ ⠹ 
B_id 2 → Home Network Equipment
⚠ Ollama stderr: ⠙ ⠹ ⠸ 
B_id 3 → Computer Manufacturer Support
⚠ Ollama stderr: ⠋ ⠙ ⠸ 
B_id 4 → HP Computer Peripherals
⚠ Ollama stderr: ⠋ ⠙ ⠸ 
B_id 5 → Information Technology Support
⚠ Ollama stderr: ⠋ ⠙ ⠹ 
B_id 6 → Portable Computing Devices
⚠ Ollama stderr: ⠋ ⠹ ⠹ 
B_id 7 → Camera Electronic Devices
⚠ Ollama stderr: ⠋ ⠹ ⠸ 
B_id 8 → Desktop and Laptops
⚠ Ollama stderr: ⠙ ⠙ ⠸ 
B_id 9 → Laptop Motherboard Parts
⚠ Ollama stderr: ⠋ ⠹ ⠹ 
B_id 10 → HP Computers and Peripherals
⚠ Ollama stderr: ⠙ ⠙ ⠸ 
B_id 11 → Enterprise IT Services
⚠ Ollama stderr: ⠋ ⠹ ⠸ 
B_id 12 → Computer Power Supplies
⚠ Ollama stderr: ⠙ ⠙ ⠸ 
B_id 13 → External Network Attached Storage
⚠ 

,C_id,C_name,B_id,B_name
0,3,Fujitsu Support Services,19,Computer Hardware Support Services
1,37,HP Spectre X360 Laptop,10,HP Computers and Peripherals
2,31,HP Omen Gaming PC,8,Desktop and Laptops
3,23,Network Patch Cables,2,Home Network Equipment
4,4,Lenovo Support Services,3,Computer Manufacturer Support
5,12,Seagate NAS Drive,13,External Network Attached Storage
6,25,Data projector solutions,14,Professional Display Solutions
7,0,Canon QY6-0073 Print Head,15,Inkjet Printer Components
8,6,Flip Phone Cases,23,Mobile Device Accessories
9,27,Acer Aspire E Series,6,Portable Computing Devices


In [43]:
# Compute centroids for each B-level cluster
B_centroids = np.vstack([
    reduced[df_clean["B_id"] == b].mean(axis=0) 
    for b in df_clean["B_id"].unique() 
    if b != -1
])

B_labels = [b for b in df_clean["B_id"].unique() if b != -1]


In [44]:
agg_A = AgglomerativeClustering(n_clusters=None, distance_threshold=1.2)
A_ids = agg_A.fit_predict(B_centroids)


In [45]:
B_to_A = dict(zip(B_labels, A_ids))
df_clean["A_id"] = df_clean["B_id"].map(lambda b: B_to_A.get(b, -1))


In [49]:
def label_group(df, id_col, name_col, level="A"):
    name_map = {}
    for gid in df[id_col].unique():
        names = df[df[id_col] == gid][name_col].dropna().unique().tolist()

        prompt = f"""
        You are an expert in product taxonomy.
        These are subcategories: {', '.join(names)}.
        Suggest a concise top-level category name (max 3 words).
        ONLY return the category name.
        """

        label = cached_llm_label(prompt, max_words=3)
        name_map[gid] = label
    return name_map

A_name_map = label_group(df_clean, "A_id", "B_name", level="A")
df_clean["A_name"] = df_clean["A_id"].map(A_name_map)


In [50]:
df_clean[["A_id", "A_name", "B_id", "B_name", "C_id", "C_name"]].head(20)


,A_id,A_name,B_id,B_name,C_id,C_name
0,15,Computer Systems Maintenance,19,Computer Hardware Support Services,3,Fujitsu Support Services
1,0,Computer Hardware,10,HP Computers and Peripherals,37,HP Spectre X360 Laptop
2,0,Computer Hardware,8,Desktop and Laptops,31,HP Omen Gaming PC
3,19,Home Networking Devices,2,Home Network Equipment,23,Network Patch Cables
4,5,Information Technology Services,3,Computer Manufacturer Support,4,Lenovo Support Services
5,17,Network Attached Storage,13,External Network Attached Storage,12,Seagate NAS Drive
6,14,Commercial Signage Systems,14,Professional Display Solutions,25,Data projector solutions
7,16,Inkjet Printer Parts,15,Inkjet Printer Components,0,Canon QY6-0073 Print Head
8,11,Mobile Phone Accessories,23,Mobile Device Accessories,6,Flip Phone Cases
9,3,Computing Hardware,6,Portable Computing Devices,27,Acer Aspire E Series


In [ ]:
#Visualize a Tree

In [51]:
import numpy as np
from anytree import Node, RenderTree

# Unique (A,B,C) combinations
df_clusters = (
    df_clean[["A_id", "A_name", "B_id", "B_name", "C_id", "C_name"]]
    .drop_duplicates()
    .sort_values(["A_id", "B_id", "C_id"])
)

# How many products per C cluster (for leaf counts)
c_sizes = df_clean.groupby("C_id").size().to_dict()


In [52]:
A_nodes = {}
B_nodes = {}
C_nodes = {}

# A-level nodes
for row in df_clusters[["A_id", "A_name"]].drop_duplicates().itertuples(index=False):
    aid, aname = row
    A_nodes[aid] = Node(f"A: {aname}")

# B-level nodes
for row in df_clusters[["A_id", "B_id", "B_name"]].drop_duplicates().itertuples(index=False):
    aid, bid, bname = row
    B_nodes[bid] = Node(f"B: {bname}", parent=A_nodes[aid])

# C-level nodes (with product counts)
for row in df_clusters[["B_id", "C_id", "C_name"]].drop_duplicates().itertuples(index=False):
    bid, cid, cname = row
    size = c_sizes.get(cid, 0)
    C_nodes[cid] = Node(f"C: {cname} ({size} products)", parent=B_nodes[bid])


In [53]:
for aid in sorted(A_nodes.keys()):
    root = A_nodes[aid]
    print("\n" + "="*80)
    print(root.name)
    print("="*80)
    for pre, fill, node in RenderTree(root):
        print(f"{pre}{node.name}")



A: Product Classification
A: Product Classification
└── B: nan
    └── C: nan (82 products)

A: Computer Hardware
A: Computer Hardware
├── B: Desktop and Laptops
│   ├── C: HP Omen Gaming PC (13 products)
│   └── C: Toshiba Satellite Notebook (31 products)
└── B: HP Computers and Peripherals
    ├── C: HP Proliant Servers (11 products)
    ├── C: HP Pavilion Notebook 14 (11 products)
    └── C: HP Spectre X360 Laptop (19 products)

A: Computer Hardware &
A: Computer Hardware &
├── B: Desktop and Laptops
│   ├── C: Fujitsu Lifebook Laptop (13 products)
│   └── C: Lenovo All-in-One PC (23 products)
└── B: Enterprise IT Services
    └── C: Dell Mission Critical Support (21 products)

A: Computer Hardware Upgrades
A: Computer Hardware Upgrades
├── B: Server and Storage
│   └── C: Fujitsu Rack Server (28 products)
└── B: Computer Memory Upgrades
    └── C: Memory Modules & Kits (12 products)

A: Computing Hardware
A: Computing Hardware
├── B: Portable Computing Devices
│   ├── C: Acer Aspi

In [54]:
import json
from pathlib import Path

# Ensure folder exists
Path("data").mkdir(exist_ok=True)

taxonomy = []

for aid, aname in (
    df_clean[["A_id", "A_name"]]
    .drop_duplicates()
    .sort_values("A_id")
    .itertuples(index=False)
):
    a_entry = {
        "A_id": int(aid),
        "A_name": aname,
        "children": []
    }

    # B children
    b_rows = (
        df_clean[df_clean["A_id"] == aid][["B_id", "B_name"]]
        .drop_duplicates()
        .sort_values("B_id")
    )

    for bid, bname in b_rows.itertuples(index=False):
        b_entry = {
            "B_id": int(bid),
            "B_name": bname,
            "children": []
        }

        # C children
        c_rows = (
            df_clean[df_clean["B_id"] == bid][["C_id", "C_name"]]
            .drop_duplicates()
            .sort_values("C_id")
        )

        for cid, cname in c_rows.itertuples(index=False):
            b_entry["children"].append({
                "C_id": int(cid),
                "C_name": cname,
                "num_products": int(df_clean[df_clean["C_id"] == cid].shape[0])
            })

        a_entry["children"].append(b_entry)

    taxonomy.append(a_entry)

# Save JSON
json_path = "data/predicted_taxonomy.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(taxonomy, f, indent=2, ensure_ascii=False)

print(f"Saved hierarchical JSON → {json_path}")


Saved hierarchical JSON → data/predicted_taxonomy.json


In [55]:
# Create flat A-B-C mapping table
df_flat = (
    df_clean[["A_id", "A_name", "B_id", "B_name", "C_id", "C_name"]]
    .drop_duplicates()
    .sort_values(["A_id", "B_id", "C_id"])
)

# Add product count per C cluster
c_sizes = df_clean.groupby("C_id").size().to_dict()
df_flat["num_products"] = df_flat["C_id"].map(c_sizes)

# Save CSV
csv_path = "data/predicted_taxonomy.csv"
df_flat.to_csv(csv_path, index=False)

print(f"Saved flat CSV → {csv_path}")


Saved flat CSV → data/predicted_taxonomy.csv


In [56]:
# Example path: "Computers & Electronics>Warranty & Support>Warranty & Support Extensions"

def split_expert_path(path: str):
    if not isinstance(path, str) or not path.strip():
        return (None, None, None)
    parts = [p.strip() for p in path.split(">")]
    # pad to length 3
    while len(parts) < 3:
        parts.append(None)
    return parts[0], parts[1], parts[2]

# Apply to your dataframe
df_eval = df_clean.copy()

df_eval[["A_gt", "B_gt", "C_gt"]] = df_eval["pathlist_names"].apply(
    lambda p: pd.Series(split_expert_path(p))
)

df_eval[["pathlist_names", "A_gt", "B_gt", "C_gt"]].head(10)


,pathlist_names,A_gt,B_gt,C_gt
0,Computers & Electronics>Warranty & Support>War...,Computers & Electronics,Warranty & Support,Warranty & Support Extensions
1,Computers & Electronics>Computers>Notebooks,Computers & Electronics,Computers,Notebooks
2,Computers & Electronics>Computers>PCs/Workstat...,Computers & Electronics,Computers,PCs/Workstations
3,Computers & Electronics>Computer Cables>Networ...,Computers & Electronics,Computer Cables,Networking Cables
4,Computers & Electronics>Warranty & Support>War...,Computers & Electronics,Warranty & Support,Warranty & Support Extensions
5,Computers & Electronics>Data Storage>Data Stor...,Computers & Electronics,Data Storage,Data Storage Devices
6,Computers & Electronics>Consumer Audio & Video...,Computers & Electronics,Consumer Audio & Video Equipment,Audio Equipment Parts & Accessories
7,Computers & Electronics>Printers & Scanners>Pr...,Computers & Electronics,Printers & Scanners,Print & Scan Accessories
8,Computers & Electronics>Telecom & Navigation>M...,Computers & Electronics,Telecom & Navigation,Mobile Phone Cases
9,Computers & Electronics>Computers>Notebooks,Computers & Electronics,Computers,Notebooks


Coverage (how many products you placed)

We’ll measure:

Coverage for all products

Coverage for products that have a ground-truth path

In [57]:
import numpy as np

# 1) Overall coverage (C-level)
total_products = len(df_eval)
placed_products = (df_eval["C_id"] != -1).sum()
coverage_all = placed_products / total_products

# 2) Coverage among products that have an expert path
has_gt = df_eval["C_gt"].notna()
total_with_gt = has_gt.sum()
placed_with_gt = ((df_eval["C_id"] != -1) & has_gt).sum()
coverage_gt = placed_with_gt / total_with_gt if total_with_gt > 0 else np.nan

print(f"Total products: {total_products}")
print(f"C-level placed products: {placed_products}  → Coverage (all) = {coverage_all:.3f}")
print(f"Products with GT leaf (C_gt): {total_with_gt}")
print(f"C-level placed among GT: {placed_with_gt} → Coverage (with GT) = {coverage_gt:.3f}")



Total products: 1000
C-level placed products: 918  → Coverage (all) = 0.918
Products with GT leaf (C_gt): 1000
C-level placed among GT: 918 → Coverage (with GT) = 0.918


NMI between predicted clusters and expert leaf categories

For NMI we need two integer label arrays:

predicted cluster id → C_id

expert leaf category id → encode C_gt as integers

In [58]:
from sklearn.metrics import normalized_mutual_info_score

# Only evaluate on rows that have a GT leaf and a non-noise cluster
mask_nmi = has_gt & (df_eval["C_id"] != -1)

pred_labels = df_eval.loc[mask_nmi, "C_id"].astype(int)

# Encode C_gt strings as integers
gt_leaf_str = df_eval.loc[mask_nmi, "C_gt"].astype(str)
gt_leaf_codes, gt_leaf_uniques = pd.factorize(gt_leaf_str)

print(f"Unique GT leaf categories used in NMI: {len(gt_leaf_uniques)}")

nmi_c = normalized_mutual_info_score(gt_leaf_codes, pred_labels)
print(f"NMI between predicted C_id and Icecat C_gt: {nmi_c:.3f}")


Unique GT leaf categories used in NMI: 84
NMI between predicted C_id and Icecat C_gt: 0.702


In [59]:
pd.crosstab(
    df_eval.loc[mask_nmi, "C_name"],
    df_eval.loc[mask_nmi, "C_gt"]
).head(15)


C_gt,AV Extenders,All-in-One PCs/Workstations,Antivirus Security Software,Audio Equipment Parts & Accessories,Battery Chargers,Calculators,Camera Accessories,Cameras & Camcorders,Chassis Components,Composite Video Cables,...,TV Set-Top Boxes,TVs,Tablet Cases,Tablet Spare Parts,Tablets,UPS Batteries,USB Cables,Uninterruptible Power Supplies (UPSs),Warranty & Support Extensions,Wireless Routers
C_name,,,,,,,,,,,,,,,,,,,,,
1. Zyxel Modem 2.,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,2
Acer Aspire E Series,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Acer Notebook Motherboard Spares,0,0,0,0,0,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
Antivirus Security Software,0,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Asus Laptops 6th Gen,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Canon QY6-0073 Print Head,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Data projector solutions,0,0,0,6,0,0,3,0,1,0,...,1,23,0,0,0,0,0,0,0,0
Dell Mission Critical Support,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,3,0
Desktop Gaming PC,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [71]:
from anytree import Node as AnyNode, RenderTree  # for visualization (we already used this)
from zss import Node as ZssNode, simple_distance  # for TED


In [72]:
from zss import Node as ZssNode

def build_zss_tree_from_triplets(triplets, col_A, col_B, col_C):
    """
    triplets: DataFrame with columns [col_A, col_B, col_C]
    Returns: root ZssNode of a tree representing A > B > C hierarchy.
    """
    root = ZssNode("ROOT")
    a_nodes = {}
    b_nodes = {}

    for row in triplets.itertuples(index=False):
        A = getattr(row, col_A)
        B = getattr(row, col_B)
        C = getattr(row, col_C)

        # skip rows with missing values
        if not isinstance(A, str) or not isinstance(B, str) or not isinstance(C, str):
            continue

        # A-level (child of ROOT)
        if A not in a_nodes:
            a_node = ZssNode(A)
            root.addkid(a_node)
            a_nodes[A] = a_node

        # B-level (child of A)
        key_B = (A, B)
        if key_B not in b_nodes:
            b_node = ZssNode(B)
            a_nodes[A].addkid(b_node)
            b_nodes[key_B] = b_node

        # C-level (child of B)
        c_node = ZssNode(C)
        b_nodes[key_B].addkid(c_node)

    return root


In [73]:
# Ground-truth tree (Icecat)
gt_triplets = (
    df_eval[["A_gt", "B_gt", "C_gt"]]
    .dropna()
    .drop_duplicates()
)

gt_root = build_zss_tree_from_triplets(gt_triplets, "A_gt", "B_gt", "C_gt")

# Predicted tree (your taxonomy)
pred_triplets = (
    df_eval[["A_name", "B_name", "C_name"]]
    .dropna()
    .drop_duplicates()
)

pred_root = build_zss_tree_from_triplets(pred_triplets, "A_name", "B_name", "C_name")


In [74]:
from zss import simple_distance

ted_value = simple_distance(gt_root, pred_root)
print("Tree Edit Distance (predicted vs expert):", ted_value)


Tree Edit Distance (predicted vs expert): 137.0


In [75]:
def count_nodes_zss(node):
    return 1 + sum(count_nodes_zss(c) for c in node.children)

gt_nodes = count_nodes_zss(gt_root)
pred_nodes = count_nodes_zss(pred_root)
max_nodes = gt_nodes + pred_nodes

ted_norm = ted_value / max_nodes if max_nodes > 0 else 0.0

print(f"TED raw: {ted_value}")
print(f"GT nodes: {gt_nodes}, Pred nodes: {pred_nodes}")
print(f"Normalized TED: {ted_norm:.4f}")


TED raw: 137.0
GT nodes: 107, Pred nodes: 85
Normalized TED: 0.7135
